# TAF — Colab runner

Runtime → Change runtime type → **GPU**. Run top to bottom. Results are written to `My Drive/TAF_Project/results`, and every sweep is resumable: if Colab disconnects, re-run the same cell and finished runs are skipped.

**Order:** smoke test → `pilot` → (read summary, fix TAF_DEFAULT after `tune`) → other groups.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/TAF_Project'
RES = f'{PROJ}/results'
!mkdir -p $RES

### Get the code
Unpacks `TAF-FederatedLearning.zip` from `My Drive/TAF_Project`.

In [ ]:
import os
REPO = '/content/TAF-FederatedLearning'
ZIP = '/content/drive/MyDrive/TAF_Project/TAF-FederatedLearning.zip'
assert os.path.exists(ZIP), 'Upload TAF-FederatedLearning.zip into My Drive/TAF_Project first'
!rm -rf $REPO && unzip -q -o $ZIP -d /content/
%cd $REPO
!git log --oneline -1

In [ ]:
!pip -q install -r requirements.txt

### 1. Smoke test (≈1 min)
Checks that the code satisfies Lemma 1 and Propositions 1–3 and that the adaptive attack hits its target cosine.

In [ ]:
!python -m pytest tests/test_theory.py -q

### 2. Pilot (8 runs) — run this before anything else
Measures real per-run time on this GPU and checks whether honest similarities at κ=0.5 clear the Proposition 2 threshold on real data. **Send the printed summary back before starting the large groups.**

In [ ]:
!python scripts/sweep.py --group pilot --out $RES --data-root /content/data --workers 2
!python scripts/summarize_pilot.py --results $RES

### 2a. Trainer benchmark (≈5 min)
Times one round per trainer setting on this GPU. Set `TRAINER` / `CHUNK` in the next cell to the fastest line that did not run out of memory.

In [ ]:
!python scripts/bench_trainer.py --data-root /content/data

In [ ]:
TRAINER, CHUNK = 'vmap', 32   # <- edit after reading the benchmark

### 2b. Pilot 2 — redesigned TAF + adaptive attack (34 runs; finished ones are skipped)
**Paste the full printout back.**

In [ ]:
!python scripts/sweep.py --group pilot2 --out $RES --data-root /content/data --workers 1 --trainer $TRAINER --chunk $CHUNK
!python scripts/summarize.py --results $RES --group pilot2

### 3. Server overhead (minutes, no training)

In [ ]:
!python scripts/overhead.py --dataset cifar10 --n 50 100 200 --reps 10 --out $RES/overhead.json

### 4. Hyperparameter tuning (MNIST, validation split)
After this finishes, copy the selected values into `TAF_DEFAULT` in `configs/experiments.py` and commit. Only then run the groups below.

In [ ]:
!python scripts/sweep.py --group tune --out $RES --data-root /content/data --workers 3

### 5. Main experiments (long — run across several sessions)

In [ ]:
# GROUP = 'main'   # then: adaptive, noniid, rho_sweep, ablation, sensitivity
# !python scripts/sweep.py --group $GROUP --out $RES --data-root /content/data --workers 3